In [2]:
import os
import glob
import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 300


In [15]:
# ---------------------------------------------------------
# Step 1: Load and Stratify Raw Data (N = 50,000)
# ---------------------------------------------------------
DATA_DIR = "../raw_data"
csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))

print("=" * 70)
print("1. INGESTING DATA AND BUILDING STRATIFIED SAMPLE (N = 50,000)")
print("=" * 70)

# Target columns for loan evaluation
cols_needed = [
    'LoanStatus', 'BusinessAge', 'TermInMonths', 'GrossApproval',
    'JobsSupported', 'CollateralInd', 'NaicsCode', 'FranchiseCode',
    'Program', 'UrbanRural', 'ApprovalFY'
]

raw_chunks = []
for f in csv_files:
    # Read relevant columns in chunks
    df_temp = pd.read_csv(f, low_memory=False)
    # Filter matching columns
    matched_cols = [c for c in cols_needed if c in df_temp.columns]
    df_filtered = df_temp[matched_cols].dropna(subset=['LoanStatus']).copy()
    
    # Normalize LoanStatus
    df_filtered['LoanStatus'] = df_filtered['LoanStatus'].astype(str).str.strip().str.upper()
    # Filter to resolved loans only
    resolved = df_filtered[df_filtered['LoanStatus'].isin(['CHGOFF', 'PIF', 'P I F'])].copy()
    raw_chunks.append(resolved)

full_resolved_df = pd.concat(raw_chunks, ignore_index=True)
full_resolved_df['LoanStatus'] = full_resolved_df['LoanStatus'].replace({'P I F': 'PIF'})
print(f"Total resolved loans in database: {len(full_resolved_df):,}")

# Stratified random sample of 50,000 based on LoanStatus
sample_df = full_resolved_df.groupby('LoanStatus', group_keys=False).apply(
    lambda x: x.sample(n=int(50000 * len(x) / len(full_resolved_df)), random_state=42)
).reset_index(drop=True)

# Adjust to exactly 50,000 if rounding caused +/- 1 record
if len(sample_df) < 50000:
    extra = full_resolved_df.sample(50000 - len(sample_df), random_state=42)
    sample_df = pd.concat([sample_df, extra], ignore_index=True)
elif len(sample_df) > 50000:
    sample_df = sample_df.sample(50000, random_state=42).reset_index(drop=True)

print(f"Stratified Sample Size: {len(sample_df):,}")
print("Target Class Distribution:")
print(sample_df['LoanStatus'].value_counts(normalize=True) * 100)

1. INGESTING DATA AND BUILDING STRATIFIED SAMPLE (N = 50,000)
Total resolved loans in database: 1,526,662
Stratified Sample Size: 50,000
Target Class Distribution:
LoanStatus
PIF       84.748
CHGOFF    15.252
Name: proportion, dtype: float64


C:\Users\durgesh.choudhary\AppData\Local\Temp\ipykernel_47360\1219617446.py:37: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sample_df = full_resolved_df.groupby('LoanStatus', group_keys=False).apply(


In [ ]:
# ---------------------------------------------------------
# Step 2: Data Cleaning Log Diagnostics
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("2. DATA CLEANING AUDIT & PREPROCESSING")
print("=" * 70)

# A. Duplicate audit
dup_count = sample_df.duplicated().sum()
sample_df = sample_df.drop_duplicates().reset_index(drop=True)
print(f"Duplicate rows removed: {dup_count} ({(dup_count/50000)*100:.2f}%)")

# B. Missingness detection
print("\nMissing values before imputation:")
missing_series = sample_df.isnull().sum()
missing_pct = (sample_df.isnull().sum() / len(sample_df)) * 100
missing_report = pd.DataFrame({'Missing Count': missing_series, 'Percent (%)': missing_pct})
print(missing_report[missing_report['Missing Count'] > 0])

# C. Handling Missing Values and Data Types
# Clean GrossApproval to float
if sample_df['GrossApproval'].dtype == object:
    sample_df['GrossApproval'] = sample_df['GrossApproval'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False)
sample_df['GrossApproval'] = pd.to_numeric(sample_df['GrossApproval'], errors='coerce')
sample_df['GrossApproval'] = sample_df['GrossApproval'].fillna(sample_df['GrossApproval'].median())

# Clean TermInMonths
sample_df['TermInMonths'] = pd.to_numeric(sample_df['TermInMonths'], errors='coerce')
sample_df['TermInMonths'] = sample_df['TermInMonths'].fillna(sample_df['TermInMonths'].median())
# Filter invalid 0 term records
sample_df = sample_df[sample_df['TermInMonths'] > 0].copy()

# Clean JobsSupported
sample_df['JobsSupported'] = pd.to_numeric(sample_df['JobsSupported'], errors='coerce').fillna(1)

# Clean Categoricals
sample_df['BusinessAge'] = sample_df['BusinessAge'].fillna('Existing Business').astype(str)
sample_df['CollateralInd'] = sample_df['CollateralInd'].fillna('N').astype(str).str.upper()

# D. Feature Engineering
sample_df['NaicsSector'] = sample_df['NaicsCode'].astype(str).str[:2]
sample_df['FranchiseInd'] = sample_df['FranchiseCode'].astype(str).apply(
    lambda x: 0 if x in ['0', '99999', 'nan', ''] else 1
)
sample_df['Target_Default'] = (sample_df['LoanStatus'] == 'CHGOFF').astype(int)


2. DATA CLEANING AUDIT & PREPROCESSING
Duplicate rows removed: 913 (1.83%)

Missing values before imputation:
               Missing Count  Percent (%)
BusinessAge               54     0.110009
CollateralInd            543     1.106199
NaicsCode               5538    11.282009
FranchiseCode          45549    92.792389


In [17]:
# ---------------------------------------------------------
# Step 3: Statistical Hypothesis Testing (RQ1 & RQ2)
# ---------------------------------------------------------
print("\n" + "=" * 70)
print("3. HYPOTHESIS TESTING RESULTS (RQ1 & RQ2)")
print("=" * 70)

# RQ1: Chi-Square Test (BusinessAge vs LoanStatus)
print("--- Research Question 1: Chi-Square Test of Independence ---")
contingency_table = pd.crosstab(sample_df['BusinessAge'], sample_df['LoanStatus'])
print("Contingency Table:")
print(contingency_table)

chi2, p_val_rq1, dof, expected = stats.chi2_contingency(contingency_table)
n_total = contingency_table.sum().sum()
min_dim = min(contingency_table.shape) - 1
cramers_v = np.sqrt(chi2 / (n_total * min_dim))

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"Degrees of Freedom: {dof}")
print(f"p-value: {p_val_rq1:.4e}")
print(f"Cramer's V Effect Size: {cramers_v:.4f}")

# RQ2: Welch's t-Test (TermInMonths by LoanStatus)
print("\n--- Research Question 2: Independent Welch's t-Test ---")
term_chgoff = sample_df[sample_df['LoanStatus'] == 'CHGOFF']['TermInMonths'].dropna()
term_pif = sample_df[sample_df['LoanStatus'] == 'PIF']['TermInMonths'].dropna()

mean_chgoff, std_chgoff = term_chgoff.mean(), term_chgoff.std()
mean_pif, std_pif = term_pif.mean(), term_pif.std()

t_stat, p_val_rq2 = stats.ttest_ind(term_chgoff, term_pif, equal_var=False)

# Cohen's d calculation
pooled_std = np.sqrt(((len(term_chgoff) - 1) * std_chgoff**2 + (len(term_pif) - 1) * std_pif**2) / (len(term_chgoff) + len(term_pif) - 2))
cohens_d = (mean_chgoff - mean_pif) / pooled_std

print(f"CHGOFF (Default) Mean Term: {mean_chgoff:.2f} months (SD = {std_chgoff:.2f})")
print(f"PIF (Paid) Mean Term: {mean_pif:.2f} months (SD = {std_pif:.2f})")
print(f"Welch's t-Statistic: {t_stat:.4f}")
print(f"p-value: {p_val_rq2:.4e}")
print(f"Cohen's d Effect Size: {cohens_d:.4f}")


3. HYPOTHESIS TESTING RESULTS (RQ1 & RQ2)
--- Research Question 1: Chi-Square Test of Independence ---
Contingency Table:
LoanStatus                              CHGOFF    PIF
BusinessAge                                          
Change of Ownership                         36    449
Existing Business                           20     34
Existing or more than 2 years old          248   2346
Existing, 5 or more years                 1331   7545
Less than 3 years old but at least 2       330   1229
Less than 4 years old but at least 3      2973  16951
Less than 5 years old but at least 4       221    751
New Business or 2 years or less             51    352
New, Less than 1 Year old                 1616   7120
Startup, Loan Funds will Open Business     436   2801
Unanswered                                 327   1873
Chi-Square Statistic: 270.7107
Degrees of Freedom: 10
p-value: 2.3688e-52
Cramer's V Effect Size: 0.0743

--- Research Question 2: Independent Welch's t-Test ---
CHGOFF (Defau

In [ ]:
# ---------------------------------------------------------
# Step 4: Generate EDA Visualizations for Report
# ---------------------------------------------------------
os.makedirs("../screenshots", exist_ok=True)

# Figure 1: BusinessAge vs Default Rate
plt.figure(figsize=(10, 6))  
age_default_rate = sample_df.groupby('BusinessAge')['Target_Default'].mean().reset_index()

age_default_rate = age_default_rate.sort_values('Target_Default', ascending=False)

sns.barplot(data=age_default_rate, x='BusinessAge', y='Target_Default', palette='Blues_r')
plt.title("Figure 1: Default Rate by Business Lifecycle Stage", fontsize=11, fontweight='bold')
plt.ylabel("Default Rate (Proportion)")
plt.xlabel("Business Age Category")

plt.xticks(rotation=45, ha='right')

plt.tight_layout()  # Adjusts margins so the rotated labels are not cut off
plt.savefig("../screenshots/figure1_business_age_default.png")
plt.close()

# Figure 2: Boxplot of TermInMonths by LoanStatus
plt.figure(figsize=(8, 5))
sns.boxplot(data=sample_df, x='LoanStatus', y='TermInMonths', palette='Set2')
plt.title("Figure 2: Distribution of Loan Term (Months) by Loan Disposition", fontsize=11, fontweight='bold')
plt.xlabel("Loan Status")
plt.ylabel("Term (Months)")

plt.xticks(rotation=0)  

plt.tight_layout()
plt.savefig("../screenshots/figure2_loan_term_distribution.png")
plt.close()

# Figure 3: Distribution of Log(GrossApproval)
plt.figure(figsize=(8, 5))
sample_df['Log_GrossApproval'] = np.log(sample_df['GrossApproval'])
sns.histplot(data=sample_df, x='Log_GrossApproval', hue='LoanStatus', kde=True, bins=40, palette='tab10')
plt.title("Figure 3: Distribution of Log Loan Principal by Loan Disposition", fontsize=12, fontweight='bold')
plt.xlabel("Log(Gross Loan Approval Amount)")
plt.ylabel("Frequency")
plt.tight_layout()
plt.savefig("../screenshots/figure3_log_gross_approval.png")
plt.close()

print("\nAll figures exported successfully to '../screenshots/' folder.")

C:\Users\durgesh.choudhary\AppData\Local\Temp\ipykernel_47360\1759642835.py:9: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=age_default_rate, x='BusinessAge', y='Target_Default', palette='Blues_r')
C:\Users\durgesh.choudhary\AppData\Local\Temp\ipykernel_47360\1759642835.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=sample_df, x='LoanStatus', y='TermInMonths', palette='Set2')



All figures exported successfully to './figures/' folder.


In [3]:
df = pd.read_csv("../processed_data/analysis_sample.csv", low_memory=False)
df.head(10)

,Program,GrossApproval,ApprovalFY,TermInMonths,BusinessAge,LoanStatus,JobsSupported,CollateralInd,Default,NaicsSector,FranchiseInd,RevolverStatus
0,504,357000.0,1995,240.0,"New, Less than 1 Year old",PIF,10.0,N,0,NaN,NaN,NaN
1,504,191000.0,2006,120.0,Less than 4 years old but at least 3,PIF,6.0,N,0,48.0,NaN,NaN
2,504,179000.0,1997,242.0,"New, Less than 1 Year old",PIF,15.0,N,0,62.0,NaN,NaN
3,504,218000.0,2001,240.0,Less than 4 years old but at least 3,PIF,7.0,N,0,54.0,NaN,NaN
4,504,110000.0,2013,240.0,Less than 4 years old but at least 3,PIF,2.0,Y,0,71.0,NaN,NaN
5,504,156000.0,2012,240.0,Less than 4 years old but at least 3,PIF,5.0,Y,0,62.0,NaN,NaN
6,504,932000.0,2012,240.0,Less than 4 years old but at least 3,PIF,5.0,Y,0,42.0,NaN,NaN
7,504,485000.0,1999,240.0,Less than 4 years old but at least 3,PIF,7.0,N,0,NaN,NaN,NaN
8,504,629000.0,1996,240.0,"New, Less than 1 Year old",PIF,21.0,N,0,NaN,NaN,NaN
9,504,2563000.0,2011,240.0,Less than 4 years old but at least 3,PIF,22.0,Y,0,42.0,NaN,NaN
